# Mock Feenyx Data Engineering Assessment
**Role target:** Senior AWS Data Engineer (Glue / Python / SQL) · **Time limit: 90 minutes** — start a timer now.

This mirrors the real Feenyx format: a JupyterHub notebook, function stubs graded by
hidden test cases, SQL tasks graded pass/fail, an ML task scored on **MAE/RMSE**, and a
**required written explanation** for every task (on the real platform these are scored —
do not skip them).

**Rules for realism**
1. No AI assistants. Feenyx runs AI-use detection; practice like it's live.
2. Documentation lookups are OK (matches most real configurations).
3. Run the grading cell for each task as you go — partial submission beats a blank.

**Scenario:** You support a facilities telemetry platform for a launch-range customer.
You'll clean maintenance logs, flatten an API payload, aggregate for reporting, query a
telemetry database, and build a quick cost model.

| Task | Skill | Points |
|---|---|---|
| 1 | pandas cleaning + dedup | 25 |
| 2 | nested JSON / API parsing | 15 |
| 3 | join + aggregate | 15 |
| 4 | SQL window ranking | 15 |
| 5 | SQL gap detection (LAG) | 15 |
| 6 | regression, MAE/RMSE | 15 |


In [ ]:
import json
import sqlite3

import numpy as np
import pandas as pd

import grader   # local grading harness — same idea as Feenyx's hidden test cases

pd.set_option("display.max_columns", None)

---
## Task 1 — Clean the maintenance logs (25 pts)

`data/maintenance_logs_raw.csv` is a raw export riddled with real-world mess.
Write **`clean_maintenance_logs(df)`** that takes the raw DataFrame (all columns read as
strings) and returns a cleaned DataFrame with **exactly** these columns:

| column | requirement |
|---|---|
| `log_id` | unchanged |
| `asset_id` | uppercase (some rows are lowercase) |
| `event_ts` | parsed to `datetime64` — timestamps come in **4 different formats** (month-first for slash dates) |
| `status` | normalized to `operational` / `degraded` / `offline`. Rule: lowercase, strip all non-letters, then: `op` or `operational` → operational; starts with `degrad` → degraded; contains `off` → offline |
| `labor_hours` | float; strip text like `" hrs"`; empty → `NaN` |
| `cost_usd` | float; strip `$` and thousands commas |
| `technician` | unchanged, but empty strings → `NaN` |

**Deduplication:** drop exact duplicate rows, then — where the same `log_id` appears more
than once — keep only the row with the **latest `event_ts`**. Return sorted by `log_id`
(grading is order-insensitive but sorted output is good practice).

In [ ]:
raw_logs = pd.read_csv("data/maintenance_logs_raw.csv", dtype=str)
raw_logs.head(10)

In [ ]:
def clean_maintenance_logs(df):
    # YOUR CODE HERE
    ...


**Explain your solution** (required — 2–4 sentences: approach, assumptions, edge cases):

> *your explanation here*

In [ ]:
grader.grade_task1(clean_maintenance_logs)

---
## Task 2 — Flatten the work-orders API payload (15 pts)

`data/work_orders_api.json` is a response from an internal REST API. Write
**`flatten_work_orders(payload)`** taking the parsed dict and returning one row per work
order with columns:

- `work_order_id`, `asset_id`, `facility`, `priority`
- `opened_at` — parsed to datetime
- `assigned_user` — the nested `assigned_to.user`; note `assigned_to` can be `null` → `NaN`
- `parts_count` — number of items in `parts` (may be empty)
- `parts_total_cost` — `sum(qty * unit_cost)` over parts, rounded to 2 dp (0 if no parts)

In [ ]:
with open("data/work_orders_api.json") as f:
    payload = json.load(f)
payload["results"][0]

In [ ]:
def flatten_work_orders(payload):
    # YOUR CODE HERE
    ...


**Explain your solution:**

> *your explanation here*

In [ ]:
grader.grade_task2(flatten_work_orders)

---
## Task 3 — Facility reliability summary (15 pts)

Write **`facility_summary(clean_logs, assets)`** — inputs are your cleaned logs (Task 1
shape) and `data/asset_reference.csv`. Inner-join on `asset_id`, then aggregate per
`facility`:

- `total_events` — count of log rows
- `total_cost_usd` — sum of `cost_usd`, rounded 2 dp
- `avg_labor_hours` — mean ignoring NaN, rounded 3 dp
- `pct_offline` — fraction of events with status `offline`, rounded 3 dp

Return one row per facility, sorted by `facility`.

In [ ]:
assets = pd.read_csv("data/asset_reference.csv")

def facility_summary(clean_logs, assets):
    # YOUR CODE HERE
    ...


**Explain your solution:**

> *your explanation here*

In [ ]:
grader.grade_task3(facility_summary)

---
## Task 4 — SQL: top power consumers per facility (15 pts)

`data/telemetry.db` (SQLite) has tables `assets(asset_id, facility, asset_type,
commissioned_year, rated_power_kw)` and `power_readings(reading_id, asset_id, reading_ts,
power_kw, temp_c)`.

Write a query returning, **for each facility, the top 3 assets by average `power_kw`**:

- columns: `facility`, `asset_id`, `avg_power_kw` (rounded 2 dp), `facility_rank`
- use a window function (`RANK`) partitioned by facility
- order by `facility`, `facility_rank`, `asset_id`

In [ ]:
# explore the schema
conn = sqlite3.connect("data/telemetry.db")
print(pd.read_sql_query("SELECT * FROM power_readings LIMIT 5", conn))
conn.close()

In [ ]:
SQL_TASK4 = """
-- YOUR QUERY HERE
"""

grader.grade_task4(SQL_TASK4)

**Explain your query:**

> *your explanation here*

---
## Task 5 — SQL: telemetry gap detection (15 pts)

Readings arrive every 3 hours, but some are missing. A **gap** = the interval between an
asset's consecutive readings exceeds **3.0 hours** (strictly greater).

Return `asset_id`, `gap_count` for assets with **at least 5 gaps**, ordered by
`gap_count` descending, then `asset_id`. Use `LAG(...) OVER (PARTITION BY ... ORDER BY ...)`.
Hint: in SQLite, hours between two timestamps = `(julianday(t2) - julianday(t1)) * 24`.

In [ ]:
SQL_TASK5 = """
-- YOUR QUERY HERE
"""

grader.grade_task5(SQL_TASK5)

**Explain your query:**

> *your explanation here*

---
## Task 6 — Predict energy cost (15 pts, scored on RMSE)

Train on `data/energy_train.csv` (features + `energy_cost_usd` target) and predict
`energy_cost_usd` for every row of `data/energy_test_features.csv` (same row order).

Scoring against hidden labels — exactly how Feenyx does it:
- **RMSE ≤ 12.00 → full 15 pts** · RMSE ≤ 25.00 → 8 pts · else 0
- A plain linear regression on the raw features scores ~8 pts. Think about how the
  features physically combine to produce an energy cost if you want full credit.

In [ ]:
train = pd.read_csv("data/energy_train.csv")
test = pd.read_csv("data/energy_test_features.csv")
train.head()

In [ ]:
# Build your model, then produce `predictions` (array-like, len == len(test))
predictions = ...

grader.grade_task6(predictions)

**Explain your model choice and expected error:**

> *your explanation here*

---
## Final scorecard

Run when done (or when the 90 minutes are up):

In [ ]:
grader.scorecard()

### After the attempt
- `solutions.py` has reference solutions — review only after scoring yourself.
- Redo any failed task from scratch the next day; that's where the retention happens.
- On the real assessment, budget time to write every explanation cell — they're scored.